In [ ]:
# Install all dependencies
!pip install -q tensorflow opencv-python-headless matplotlib scikit-learn seaborn kaggle kagglehub


In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ── Dataset Download via KaggleHub ──────────────────────────────────────────
import kagglehub
_kaggle_path = kagglehub.dataset_download("emmarex/plantdisease")
print("Path to dataset files:", _kaggle_path)

DATASET_PATH = Path(_kaggle_path)
DATA_DIR     = DATASET_PATH / "PlantVillage"

# Verify the path
if not DATA_DIR.exists():
    print(f"[!] PlantVillage subfolder not found, listing contents of download:")
    for item in DATASET_PATH.iterdir():
        print(" ", item)
else:
    print(f"[✓] Dataset ready at {DATA_DIR}")

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 42

# ── Auto-detect ALL 38 classes from the dataset folder ──────────────────────
if DATA_DIR.exists():
    CLASSES      = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
    CLASS_LABELS = [c.replace("_", " ") for c in CLASSES]
    NUM_CLASSES  = len(CLASSES)
    print(f"[✓] Found {NUM_CLASSES} classes:")
    for c in CLASSES: print(f"    {c}")
else:
    CLASSES = CLASS_LABELS = []
    NUM_CLASSES = 0

# ── 5 representative classes for Sobel / filter visualisation ───────────────
VIS_CLASSES = [
    "Tomato_healthy",
    "Tomato_Early_blight",
    "Tomato_Late_blight",
    "Potato___Early_blight",
    "Corn_(maize)___Common_rust_",
]
VIS_CLASSES = [c for c in VIS_CLASSES if c in CLASSES]   # keep only present ones


Classes      : 5


In [ ]:
def _build_train_datagen():
    return ImageDataGenerator(
        preprocessing_function = tf.keras.applications.mobilenet_v2.preprocess_input,
        rotation_range     = 30,
        width_shift_range  = 0.15,
        height_shift_range = 0.15,
        shear_range        = 0.1,
        zoom_range         = 0.2,
        horizontal_flip    = True,
        vertical_flip      = False,
        fill_mode          = "nearest",
        validation_split   = 0.2,
    )

def _build_eval_datagen():
    return ImageDataGenerator(
        preprocessing_function = tf.keras.applications.mobilenet_v2.preprocess_input
    )

def make_generators():
    train_datagen = _build_train_datagen()
    eval_datagen  = _build_eval_datagen()
    shared = dict(directory=DATA_DIR, target_size=IMG_SIZE,
                  batch_size=BATCH_SIZE, classes=CLASSES,
                  class_mode="categorical", seed=SEED)
    train_gen = train_datagen.flow_from_directory(**shared, subset="training",   shuffle=True)
    val_gen   = train_datagen.flow_from_directory(**shared, subset="validation", shuffle=False)
    test_gen  = eval_datagen.flow_from_directory(**shared,  shuffle=False)
    return train_gen, val_gen, test_gen

def make_tf_datasets(train_gen, val_gen):
    AUTOTUNE = tf.data.AUTOTUNE
    sig = (
        tf.TensorSpec(shape=(None, *IMG_SIZE, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, NUM_CLASSES),  dtype=tf.float32),
    )
    train_ds = tf.data.Dataset.from_generator(lambda: train_gen, output_signature=sig).prefetch(AUTOTUNE)
    val_ds   = tf.data.Dataset.from_generator(lambda: val_gen,   output_signature=sig).prefetch(AUTOTUNE)
    return train_ds, val_ds

def show_samples(gen, n=10):
    images, labels = next(gen)
    # Denormalize MobileNetV2 preprocess_input ([-1,1] -> [0,1])
    images_display = (images + 1.0) / 2.0
    fig, axes = plt.subplots(2, 5, figsize=(15, 7))
    fig.suptitle("Sample Training Images (augmented)", fontsize=14)
    for idx, ax in enumerate(axes.flat):
        if idx >= n: break
        ax.imshow(np.clip(images_display[idx], 0, 1))
        ax.set_title(CLASS_LABELS[np.argmax(labels[idx])], fontsize=9)
        ax.axis("off")
    plt.tight_layout(); plt.savefig("sample_images.png", dpi=120, bbox_inches="tight"); plt.show()
    print("[✓] Saved sample_images.png")

def class_distribution(gen):
    print("\nClass distribution:")
    for cls, idx in gen.class_indices.items():
        count = sum(1 for f in gen.filenames if f.startswith(cls))
        print(f"  [{idx}] {CLASS_LABELS[idx]:<25} — {count} images")


In [ ]:
if DATA_DIR.exists():
    train_gen, val_gen, test_gen = make_generators()
    print(f"[✓] Train : {train_gen.samples} | Val : {val_gen.samples} | Test : {test_gen.samples}")
    class_distribution(train_gen)
    show_samples(train_gen)
else:
    print(f"[!] Dataset not found at {DATA_DIR}. Check your path / download step.")

[!] Dataset not found at /content/plantdisease/PlantVillage. Check your path / download step.


In [ ]:
import json

# ── Phase 1: train head with frozen base ──
EPOCHS_PHASE1 = 10
# ── Phase 2: fine-tune top layers of base ──
EPOCHS_PHASE2 = 10
LR_INITIAL    = 1e-3
LR_FINETUNE   = 1e-5

os.makedirs("models",  exist_ok=True)
os.makedirs("outputs", exist_ok=True)

def make_callbacks(monitor="val_accuracy", ckpt_path="models/best_model.keras"):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            ckpt_path, monitor=monitor,
            save_best_only=True, mode="max", verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
        tf.keras.callbacks.EarlyStopping(
            monitor=monitor, patience=5, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.CSVLogger("outputs/training_log.csv", append=True),
    ]

def compute_class_weights(classes, num_classes, data_root=str(DATA_DIR)):
    counts = {}
    for i, cls in enumerate(classes):
        p = Path(data_root) / cls
        counts[i] = len(list(p.glob("*.jpg"))) if p.exists() else 1
    total = sum(counts.values()) or 1
    return {i: (total / (num_classes * n)) if n > 0 else 1.0 for i, n in counts.items()}

def plot_history(history, title="Training History", save_path="outputs/training_history.png"):
    metrics = [("accuracy","val_accuracy","Accuracy"),
               ("loss","val_loss","Loss"),
               ("precision","val_precision","Precision")]
    fig, axes = plt.subplots(1, 3, figsize=(18,5))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for ax, (tk, vk, t) in zip(axes, metrics):
        if tk in history: ax.plot(history[tk], label=f"Train {t}", linewidth=2)
        if vk in history: ax.plot(history[vk], label=f"Val {t}",   linewidth=2, linestyle="--")
        ax.set_title(t); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches="tight"); plt.show()
    print(f"[✓] Saved {save_path}")

print("[✓] Training helpers ready")


In [ ]:
import cv2
import matplotlib.gridspec as gridspec
from numpy.lib.stride_tricks import sliding_window_view

SOBEL_X   = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=np.float32)
SOBEL_Y   = np.array([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=np.float32)
LAPLACIAN = np.array([[0,1,0],[1,-4,1],[0,1,0]],   dtype=np.float32)
SHARPEN   = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32)
EMBOSS    = np.array([[-2,-1,0],[-1,1,1],[0,1,2]], dtype=np.float32)

def manual_convolve(gray, kernel):
    kh, kw = kernel.shape; ph, pw = kh//2, kw//2
    padded = np.pad(gray, ((ph,ph),(pw,pw)), mode="constant")
    output = np.zeros_like(gray, dtype=np.float32)
    for r in range(gray.shape[0]):
        for c in range(gray.shape[1]):
            output[r,c] = np.sum(padded[r:r+kh, c:c+kw] * kernel)
    return output

def fast_convolve(gray, kernel):
    kh, kw  = kernel.shape; ph, pw = kh//2, kw//2
    padded  = np.pad(gray.astype(np.float32), ((ph,ph),(pw,pw)), mode="constant")
    windows = sliding_window_view(padded, (kh, kw))
    return (windows * kernel).sum(axis=(-2,-1))

def gaussian_kernel(size=5, sigma=1.0):
    axis = np.linspace(-(size//2), size//2, size)
    g    = np.exp(-0.5*(axis/sigma)**2)
    k    = np.outer(g, g)
    return k / k.sum()

print("[✓] Kernels & convolution helpers ready")

In [ ]:
def apply_all_filters(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    gray_u8 = (gray * 255).astype(np.uint8)
    results = {"Original": img_rgb}

    g_kern = gaussian_kernel(9, 2.0)
    results["Gaussian Blur\n(manual kernel)"] = np.clip(fast_convolve(gray, g_kern), 0, 1)
    results["Gaussian Blur\n(OpenCV)"]        = cv2.GaussianBlur(gray, (9,9), sigmaX=2.0)

    sx = fast_convolve(gray, SOBEL_X); sy = fast_convolve(gray, SOBEL_Y)
    mag = np.sqrt(sx**2 + sy**2)
    results["Sobel X\n(manual)"]   = np.abs(sx)
    results["Sobel Y\n(manual)"]   = np.abs(sy)
    results["Sobel Magnitude"]      = mag / mag.max() if mag.max() > 0 else mag

    sx_cv = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sy_cv = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag_cv = np.sqrt(sx_cv**2 + sy_cv**2)
    results["Sobel Magnitude\n(OpenCV)"] = mag_cv / mag_cv.max()

    lap = fast_convolve(gray, LAPLACIAN)
    results["Laplacian\n(manual)"] = np.abs(lap)
    blurred = fast_convolve(gray, gaussian_kernel(5, 1.4))
    log     = fast_convolve(blurred, LAPLACIAN)
    results["Laplacian of Gaussian\n(LoG)"] = np.abs(log)

    results["Canny Edges"]       = cv2.Canny(gray_u8, 50, 150) / 255.0
    results["Sharpen\n(manual)"] = np.clip(fast_convolve(gray, SHARPEN), 0, 1)
    results["Emboss\n(manual)"]  = np.clip(fast_convolve(gray, EMBOSS) + 0.5, 0, 1)
    return results

def plot_filter_grid(results, title="Filter Visualization", save_path=None):
    ncols = 4; nrows = (len(results) + ncols - 1) // ncols
    fig = plt.figure(figsize=(16, nrows*4))
    fig.suptitle(title, fontsize=15, fontweight="bold", y=1.01)
    gs  = gridspec.GridSpec(nrows, ncols, figure=fig, hspace=0.4, wspace=0.3)
    for idx, (name, img) in enumerate(results.items()):
        ax = fig.add_subplot(gs[idx//ncols, idx%ncols])
        ax.imshow(img) if img.ndim == 3 else ax.imshow(img, cmap="gray")
        ax.set_title(name, fontsize=9); ax.axis("off")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight"); print(f"[✓] Saved {save_path}")
    plt.show()

print("[✓] apply_all_filters & plot_filter_grid ready")

In [ ]:
# Apply filters to a real dataset image from each visualisation class
for _cls in VIS_CLASSES[:1]:   # show full filter bank for one class
    _imgs = sorted((DATA_DIR / _cls).glob("*.jpg"))
    if _imgs:
        img_bgr = cv2.imread(str(_imgs[0]))
        img_bgr = cv2.resize(img_bgr, IMG_SIZE)
        results_demo = apply_all_filters(img_bgr)
        plot_filter_grid(results_demo,
                         title=f"Filter Bank — {_cls.replace('_',' ')}",
                         save_path="filter_grid.png")
        print(f"[✓] Filters applied to real leaf image: {_imgs[0].name}")
        break


## 🏗️ Phase 3a — CNN from Scratch (Baseline)

In [ ]:
# CNN scratch needs rescale=1/255, NOT mobilenet preprocess_input
_scratch_train_datagen = ImageDataGenerator(
    rescale            = 1.0 / 255,
    rotation_range     = 30,
    width_shift_range  = 0.15,
    height_shift_range = 0.15,
    shear_range        = 0.1,
    zoom_range         = 0.2,
    horizontal_flip    = True,
    fill_mode          = "nearest",
    validation_split   = 0.2,
)
_scratch_eval_datagen = ImageDataGenerator(rescale=1.0/255)
_shared = dict(directory=DATA_DIR, target_size=IMG_SIZE,
               batch_size=BATCH_SIZE, classes=CLASSES,
               class_mode="categorical", seed=SEED)
train_gen_scratch = _scratch_train_datagen.flow_from_directory(**_shared, subset="training",   shuffle=True)
val_gen_scratch   = _scratch_train_datagen.flow_from_directory(**_shared, subset="validation", shuffle=False)
test_gen_scratch  = _scratch_eval_datagen.flow_from_directory(**_shared,  shuffle=False)
print(f"[✓] Scratch generators ready — train:{train_gen_scratch.samples} val:{val_gen_scratch.samples}")


In [ ]:
# ════════════════════════════════════════════════════════
# CNN FROM SCRATCH — Architecture
# ════════════════════════════════════════════════════════
from tensorflow.keras import layers, models, regularizers as reg_scratch

def _conv_bn_relu_scratch(x, filters, l2=1e-4):
    x = layers.Conv2D(filters, (3,3), padding="same",
                      kernel_regularizer=reg_scratch.l2(l2), use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation("relu")(x)

def conv_block_scratch(x, filters, dropout_rate=0.25, l2=1e-4):
    x = _conv_bn_relu_scratch(x, filters, l2)
    x = _conv_bn_relu_scratch(x, filters, l2)
    x = layers.MaxPooling2D((2,2))(x)
    return layers.Dropout(dropout_rate)(x)

def build_cnn_scratch(num_classes=NUM_CLASSES, input_shape=(224,224,3), l2=1e-4):
    inputs = layers.Input(shape=input_shape, name="leaf_input_scratch")
    x = conv_block_scratch(inputs, 32,  dropout_rate=0.25, l2=l2)
    x = conv_block_scratch(x,      64,  dropout_rate=0.25, l2=l2)
    x = conv_block_scratch(x,      128, dropout_rate=0.40, l2=l2)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=reg_scratch.l2(l2), name="dense_features")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="disease_class")(x)
    return models.Model(inputs=inputs, outputs=outputs, name="PlantDiseaseNet_Scratch")

cnn_scratch = build_cnn_scratch(num_classes=NUM_CLASSES)
cnn_scratch.compile(
    optimizer = tf.keras.optimizers.Adam(1e-3),
    loss      = "categorical_crossentropy",
    metrics   = ["accuracy",
                 tf.keras.metrics.Precision(name="precision"),
                 tf.keras.metrics.Recall(name="recall")],
)
cnn_scratch.summary()
print(f"\n[✓] CNN Scratch params : {cnn_scratch.count_params():,}")


In [ ]:
import time

# ── Train CNN from scratch ───────────────────────────────────────────────────
print("\n" + "="*60)
print("CNN FROM SCRATCH — Training")
print("="*60)

_t_scratch = time.time()
history_scratch = cnn_scratch.fit(
    train_gen_scratch,
    epochs          = 20,
    validation_data = val_gen_scratch,
    callbacks       = make_callbacks(ckpt_path="models/best_model_scratch.keras"),
    class_weight    = compute_class_weights(CLASSES, NUM_CLASSES),
    verbose         = 1,
)
TIME_SCRATCH    = time.time() - _t_scratch
VAL_ACC_SCRATCH = max(history_scratch.history["val_accuracy"])
PARAMS_SCRATCH  = cnn_scratch.count_params()
print(f"\n[✓] Scratch | Best val_acc: {VAL_ACC_SCRATCH:.4f} | Params: {PARAMS_SCRATCH:,} | Time: {TIME_SCRATCH:.1f}s")

plot_history(history_scratch.history,
             title="CNN from Scratch — Training History",
             save_path="outputs/training_history_scratch.png")


## 🧠 Phase 3b — MobileNetV2 Transfer Learning

In [ ]:
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.utils import plot_model

L2_DEFAULT = 1e-4

def build_model(num_classes=NUM_CLASSES, input_shape=(224, 224, 3), l2=L2_DEFAULT):
    """
    MobileNetV2 Transfer Learning model.
    Phase 1 — base frozen  : train only the new head
    Phase 2 — fine-tuning  : unfreeze top layers of base
    """
    # ── Base model (pretrained on ImageNet, no top) ──
    base = MobileNetV2(input_shape=input_shape, include_top=False, weights="imagenet")
    base.trainable = False          # freeze all base layers initially
    base._name = "mobilenet_v2_base"

    # ── Custom classification head ──
    inputs  = layers.Input(shape=input_shape, name="leaf_input")
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D(name="gap")(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(256, activation="relu",
                           kernel_regularizer=regularizers.l2(l2),
                           name="dense_features")(x)
    x       = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="disease_class")(x)

    return models.Model(inputs=inputs, outputs=outputs, name="PlantDiseaseNet_MobileV2")

def compile_model(model, learning_rate=1e-3):
    model.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss      = "categorical_crossentropy",
        metrics   = ["accuracy",
                     tf.keras.metrics.Precision(name="precision"),
                     tf.keras.metrics.Recall(name="recall")],
    )
    return model

def unfreeze_top_layers(model, num_layers=30, learning_rate=1e-5):
    """Unfreeze the top `num_layers` of MobileNetV2 for fine-tuning."""
    base = model.get_layer("mobilenet_v2_base")
    base.trainable = True
    # Freeze all except the last num_layers
    for layer in base.layers[:-num_layers]:
        layer.trainable = False
    # Recompile with a much lower LR to avoid destroying learned weights
    model.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss      = "categorical_crossentropy",
        metrics   = ["accuracy",
                     tf.keras.metrics.Precision(name="precision"),
                     tf.keras.metrics.Recall(name="recall")],
    )
    trainable = sum(1 for l in base.layers if l.trainable)
    print(f"[✓] Unfrozen top {trainable} layers of MobileNetV2 base for fine-tuning")
    return model

net = build_model()
compile_model(net)
net.summary()
print(f"\n[✓] Total params : {net.count_params():,}")
print(f"[i] MobileNetV2 base is FROZEN — only the head will train in Phase 1")


In [ ]:
# Optional: plot model diagram
# !apt-get install -q graphviz && pip install -q pydot
try:
    plot_model(net, to_file="model_architecture.png", show_shapes=True, show_layer_names=True, dpi=100)
    from IPython.display import Image; Image("model_architecture.png")
except Exception as e:
    print(f"[i] Diagram skipped: {e}")

## 🏋️ Phase 4 — Two-Phase Training (Frozen → Fine-tune)


In [ ]:
import time

class_weights = compute_class_weights(CLASSES, NUM_CLASSES)
print(f"Class weights computed for {len(class_weights)} classes\n")

# ════════════════════════════════════════════════════════
# PHASE 1 — Train head only (base frozen)
# ════════════════════════════════════════════════════════
print("\n" + "="*60)
print("PHASE 1: Training classification head (base frozen)")
print("="*60)

model = build_model(num_classes=NUM_CLASSES)
compile_model(model, learning_rate=LR_INITIAL)
model.summary()

_t0 = time.time()
history1 = model.fit(
    train_gen,
    epochs          = EPOCHS_PHASE1,
    validation_data = val_gen,
    callbacks       = make_callbacks(ckpt_path="models/best_model_phase1.keras"),
    class_weight    = class_weights,
    verbose         = 1,
)
TIME_FROZEN = time.time() - _t0
VAL_ACC_FROZEN = max(history1.history["val_accuracy"])
print(f"\n[✓] Phase 1 done | Best val_acc: {VAL_ACC_FROZEN:.4f} | Time: {TIME_FROZEN:.1f}s")

plot_history(history1.history,
             title="Phase 1 — Head Training (Frozen Base)",
             save_path="outputs/training_history_phase1.png")

# ════════════════════════════════════════════════════════
# PHASE 2 — Fine-tune top layers of MobileNetV2
# ════════════════════════════════════════════════════════
print("\n" + "="*60)
print("PHASE 2: Fine-tuning top 30 layers of MobileNetV2")
print("="*60)

model = unfreeze_top_layers(model, num_layers=30, learning_rate=LR_FINETUNE)

_t1 = time.time()
history2 = model.fit(
    train_gen,
    epochs          = EPOCHS_PHASE2,
    validation_data = val_gen,
    callbacks       = make_callbacks(ckpt_path="models/best_model.keras"),
    class_weight    = class_weights,
    verbose         = 1,
)
TIME_FINETUNE = time.time() - _t1
VAL_ACC_FINETUNE = max(history2.history["val_accuracy"])
print(f"\n[✓] Phase 2 done | Best val_acc: {VAL_ACC_FINETUNE:.4f} | Time: {TIME_FINETUNE:.1f}s")

plot_history(history2.history,
             title="Phase 2 — Fine-tuning MobileNetV2",
             save_path="outputs/training_history_phase2.png")

print("\n[✓] Two-phase training complete. Best model saved to models/best_model.keras")


In [ ]:
# Combine both phase histories for saving
combined_history = {}
for k in history1.history:
    combined_history[k] = history1.history[k] + history2.history.get(k, [])

serialised = {k: [float(v) for v in vals] for k, vals in combined_history.items()}
with open("outputs/history.json", "w") as fh:
    json.dump(serialised, fh, indent=2)
print("[✓] Saved outputs/history.json")

# Evaluate on test set
best_model = tf.keras.models.load_model("models/best_model.keras")
test_loss, test_acc, *_ = best_model.evaluate(test_gen, verbose=1)
print(f"Test accuracy : {test_acc:.4f}")
print(f"Test loss     : {test_loss:.4f}")

plot_history(combined_history,
             title="Full Training History (Phase 1 + Phase 2)",
             save_path="outputs/training_history_combined.png")

# Alias for rest of notebook
model = best_model

# Download trained model (Colab only)
# from google.colab import files; files.download("models/best_model.keras")


## 📊 Phase 4b — Feature Map Visualisation

In [ ]:
# ════════════════════════════════════════════════════════
# Feature Map Visualisation — First Conv2D Layer
# ════════════════════════════════════════════════════════
def get_feature_maps(mdl, img_bgr, layer_index=0):
    """Extract feature maps from the first Conv2D layer."""
    # Find first Conv2D
    conv_layers = [l for l in mdl.layers if isinstance(l, tf.keras.layers.Conv2D)]
    if not conv_layers:
        print("[!] No Conv2D layers found in model (MobileNetV2 uses sub-model). Using scratch CNN.")
        return None, None
    target_layer = conv_layers[layer_index]
    feat_model   = tf.keras.Model(inputs=mdl.inputs, outputs=target_layer.output)
    img_rgb = cv2.cvtColor(cv2.resize(img_bgr, IMG_SIZE), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    feat_maps = feat_model.predict(np.expand_dims(img_rgb, 0), verbose=0)[0]
    return feat_maps, target_layer.name

def plot_feature_maps(feat_maps, layer_name, title_prefix="", n=16):
    n = min(n, feat_maps.shape[-1])
    cols = 8; rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2, rows*2))
    fig.suptitle(f"{title_prefix} — {layer_name} feature maps (first {n})", fontsize=11)
    for idx in range(rows * cols):
        ax = axes[idx//cols, idx%cols] if rows > 1 else axes[idx%cols]
        if idx < n:
            fm = feat_maps[:,:,idx]
            ax.imshow(fm, cmap="viridis")
        ax.axis("off")
    plt.tight_layout()
    fname = f"outputs/feature_maps_{title_prefix.replace(' ','_').lower()}.png"
    plt.savefig(fname, dpi=100, bbox_inches="tight")
    plt.show()
    print(f"[✓] Saved {fname}")

# Get one healthy and one diseased image
_healthy_dir  = DATA_DIR / "Tomato_healthy"
_disease_dir  = DATA_DIR / "Tomato_Early_blight"
if not _healthy_dir.exists():  _healthy_dir  = next((DATA_DIR/c for c in CLASSES if "healthy" in c.lower()), None)
if not _disease_dir.exists():  _disease_dir  = next((DATA_DIR/c for c in CLASSES if "blight"  in c.lower()), None)

_healthy_imgs = sorted(_healthy_dir.glob("*.jpg")) if _healthy_dir else []
_disease_imgs = sorted(_disease_dir.glob("*.jpg")) if _disease_dir else []

if _healthy_imgs and _disease_imgs:
    img_healthy = cv2.imread(str(_healthy_imgs[0]))
    img_disease = cv2.imread(str(_disease_imgs[0]))

    # Use CNN scratch (has accessible Conv2D)
    fm_healthy, lname = get_feature_maps(cnn_scratch, img_healthy, layer_index=0)
    fm_disease, _     = get_feature_maps(cnn_scratch, img_disease, layer_index=0)

    if fm_healthy is not None:
        plot_feature_maps(fm_healthy, lname, title_prefix="Healthy Leaf")
        plot_feature_maps(fm_disease, lname, title_prefix="Diseased Leaf (Early Blight)")
        print("[i] Compare: diseased leaves activate more edge/texture-sensitive filters")
else:
    print("[!] Could not find healthy/diseased images for feature map visualisation")


## 📈 Phase 4c — Model Comparison

In [ ]:
# ════════════════════════════════════════════════════════
# MODEL COMPARISON TABLE
# ════════════════════════════════════════════════════════
import pandas as pd

comparison_data = {
    "Model"         : ["CNN from Scratch", "MobileNetV2 Frozen (Phase 1)", "MobileNetV2 Fine-tuned (Phase 2)"],
    "Val Accuracy"  : [f"{VAL_ACC_SCRATCH:.4f}", f"{VAL_ACC_FROZEN:.4f}", f"{VAL_ACC_FINETUNE:.4f}"],
    "Parameters"    : [f"{PARAMS_SCRATCH:,}",
                       f"{model.count_params():,}",
                       f"{model.count_params():,}"],
    "Training Time" : [f"{TIME_SCRATCH:.1f}s", f"{TIME_FROZEN:.1f}s", f"{TIME_FINETUNE:.1f}s"],
}

df = pd.DataFrame(comparison_data)
print("\n" + "="*70)
print("MODEL COMPARISON SUMMARY")
print("="*70)
print(df.to_string(index=False))
print("="*70)

# Visual bar chart
fig, ax = plt.subplots(figsize=(9, 4))
val_accs = [float(VAL_ACC_SCRATCH), float(VAL_ACC_FROZEN), float(VAL_ACC_FINETUNE)]
colors   = ["#e74c3c", "#3498db", "#2ecc71"]
bars = ax.bar(comparison_data["Model"], val_accs, color=colors, width=0.5, edgecolor="none")
for bar, acc in zip(bars, val_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{acc:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Validation Accuracy")
ax.set_title("Model Comparison: CNN Scratch vs MobileNetV2 Frozen vs Fine-tuned", fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("[✓] Saved outputs/model_comparison.png")


## 📊 Phase 5 — Evaluation

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

os.makedirs("outputs/annotated", exist_ok=True)

def _preprocess(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, IMG_SIZE).astype(np.float32)
    return tf.keras.applications.mobilenet_v2.preprocess_input(resized)

def predict(mdl, img_bgr):
    probs = mdl.predict(np.expand_dims(_preprocess(img_bgr), 0), verbose=0)[0]
    idx   = int(np.argmax(probs))
    return CLASS_LABELS[idx], float(probs[idx]), probs

def annotate_image(img_bgr, label, confidence, probs):
    h, w  = img_bgr.shape[:2]; out = img_bgr.copy()
    color = (0,200,0) if label == "Healthy" else (0,60,220)
    cv2.rectangle(out, (0,0), (w,40), (0,0,0), -1)
    cv2.putText(out, f"{label}  ({confidence*100:.1f}%)", (8,27),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2, cv2.LINE_AA)
    bar_area = 18*len(CLASS_LABELS)+10
    cv2.rectangle(out, (0,h-bar_area), (w,h), (20,20,20), -1)
    for i, (cls, prob) in enumerate(zip(CLASS_LABELS, probs)):
        y = h-bar_area+8+i*18; bar_w = int(prob*(w-90))
        bc = (0,180,0) if cls == "Healthy" else (0,80,200)
        cv2.rectangle(out, (85,y), (85+bar_w, y+12), bc, -1)
        cv2.putText(out, cls[:8], (2,y+10), cv2.FONT_HERSHEY_SIMPLEX, 0.32, (200,200,200), 1)
        cv2.putText(out, f"{prob*100:.1f}%", (85+bar_w+3,y+10), cv2.FONT_HERSHEY_SIMPLEX, 0.32, (200,200,200), 1)
    return out

print("[✓] Evaluation helpers ready")

In [ ]:
def evaluate_unseen_images(mdl, image_paths, true_labels=None):
    print(f"\n--- Evaluating {len(image_paths)} unseen images ---")
    y_pred, y_conf = [], []
    fig, axes = plt.subplots(4, 5, figsize=(20,16))
    fig.suptitle("Predictions on 20 Unseen Leaf Images", fontsize=14)
    for i, path in enumerate(image_paths[:20]):
        img = cv2.imread(str(path))
        if img is None: continue
        img = cv2.resize(img, IMG_SIZE)
        label, conf, probs = predict(mdl, img)
        y_pred.append(CLASS_LABELS.index(label)); y_conf.append(conf)
        annotated = annotate_image(img, label, conf, probs)
        cv2.imwrite(f"outputs/annotated/pred_{i:02d}_{label.replace(' ','_')}.jpg", annotated)
        ax = axes[i//5, i%5]
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{label}\n{conf*100:.1f}%", fontsize=8,
                     color="green" if label=="Healthy" else "red")
        ax.axis("off")
    plt.tight_layout()
    plt.savefig("outputs/annotated_predictions_grid.png", dpi=100, bbox_inches="tight"); plt.show()
    print("[✓] Saved outputs/annotated_predictions_grid.png")
    return {"y_true": true_labels, "y_pred": y_pred, "confidences": y_conf}

def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1,2,figsize=(16,6))
    fig.suptitle("Confusion Matrix", fontsize=14, fontweight="bold")
    for ax, data, fmt, title in zip(axes, [cm,cm_norm], ["d",".2f"], ["Raw Counts","Normalized (row %)"]):
        sns.heatmap(data, annot=True, fmt=fmt, cmap="YlOrRd",
                    xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, linewidths=0.5, ax=ax)
        ax.set_title(title); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.tick_params(axis="x", rotation=30, labelsize=9)
        ax.tick_params(axis="y", rotation=0,  labelsize=9)
    plt.tight_layout()
    plt.savefig("outputs/confusion_matrix.png", dpi=120, bbox_inches="tight"); plt.show()
    print("[✓] Saved outputs/confusion_matrix.png")

print("[✓] evaluate_unseen_images & plot_confusion_matrix ready")

In [ ]:
DATA_DIR_EVAL = DATA_DIR  # uses the kagglehub-downloaded path
test_images, true_labels, sobel_images = [], [], {}

for cls_idx, cls in enumerate(CLASSES):
    cls_dir = DATA_DIR_EVAL / cls
    if not cls_dir.exists(): continue
    imgs = sorted(cls_dir.glob("*.jpg"))
    # take last 4 images per class as unseen test images
    for p in imgs[-4:]:
        test_images.append(p); true_labels.append(cls_idx)
    # collect one image per VIS_CLASS for sobel grid
    if cls in VIS_CLASSES and imgs:
        sobel_images[CLASS_LABELS[CLASSES.index(cls)]] = imgs[-1]

if test_images:
    eval_results = evaluate_unseen_images(model, test_images, true_labels)
    if eval_results["y_true"] and eval_results["y_pred"]:
        plot_confusion_matrix(eval_results["y_true"], eval_results["y_pred"])
        print("\nClassification Report:")
        print(classification_report(eval_results["y_true"], eval_results["y_pred"], target_names=CLASS_LABELS))
else:
    print("[!] No test images found. Check dataset path.")


In [ ]:
def plot_sobel_across_classes(class_image_paths):
    n   = len(class_image_paths)
    fig, axes = plt.subplots(3, n, figsize=(4*n, 12))
    fig.suptitle("Sobel Filter Response Across Disease Classes", fontsize=12)
    for col, (label, path) in enumerate(class_image_paths.items()):
        img = cv2.imread(str(path))
        if img is None: continue
        img  = cv2.resize(img, IMG_SIZE)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
        sx = fast_convolve(gray, SOBEL_X); sy = fast_convolve(gray, SOBEL_Y)
        mag = np.sqrt(sx**2+sy**2); mag = mag/mag.max() if mag.max()>0 else mag
        blurred = fast_convolve(gray, gaussian_kernel(5,1.4))
        log = np.abs(fast_convolve(blurred, LAPLACIAN)); log = log/log.max() if log.max()>0 else log
        axes[0,col].imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB)); axes[0,col].set_title(label,fontsize=9,fontweight="bold"); axes[0,col].axis("off")
        axes[1,col].imshow(mag,cmap="hot");  axes[1,col].set_title("Sobel Magnitude",fontsize=8); axes[1,col].axis("off")
        axes[2,col].imshow(log,cmap="gray"); axes[2,col].set_title("LoG",fontsize=8);             axes[2,col].axis("off")
    for ax, lbl in zip(axes[:,0], ["Original","Sobel Magnitude","LoG"]):
        ax.set_ylabel(lbl, fontsize=10, rotation=90, labelpad=10)
    plt.tight_layout()
    plt.savefig("outputs/sobel_class_comparison.png", dpi=120, bbox_inches="tight"); plt.show()
    print("[✓] Saved outputs/sobel_class_comparison.png")

if sobel_images:
    plot_sobel_across_classes(sobel_images)

## 🔬 Sobel Filter Discussion — Texture Differences Across Disease Classes

The Sobel magnitude map highlights **edge density and texture coarseness** in each leaf:

| Class | Sobel / LoG pattern | What it indicates |
|---|---|---|
| **Healthy** | Low, uniform edge response; smooth leaf surface | Intact cell structure, no lesions |
| **Early Blight** | Moderate circular high-response rings | Concentric lesion boundaries visible as strong edges |
| **Late Blight** | Large irregular high-response blobs | Water-soaked lesion boundaries with diffuse edges |
| **Potato Early Blight** | Similar ring pattern to tomato early blight | Alternaria fungal lesion with distinct margins |
| **Common Rust (Corn)** | Scattered fine high-response dots | Small pustule edges distributed across the blade |

**Key observations:**
- Diseased leaves consistently show **higher Sobel magnitude** than healthy ones — disease lesions create sharp intensity transitions at lesion boundaries.
- The **LoG (Laplacian of Gaussian)** is especially sensitive to the fine texture of rust pustules and early-blight ring edges.
- Healthy leaves show strong Sobel response only at **leaf veins and outer edges**, whereas diseased leaves show additional internal responses.


## 🔍 Phase 6 — Prediction + Filter Visualization

In [ ]:
# Generate a distinct colour for each of the 38 classes
import matplotlib.cm as _cm
CLASS_COLORS = ["#%02x%02x%02x" % tuple(int(c*255) for c in _cm.tab20(i % 20)[:3]) for i in range(NUM_CLASSES)]
FILTER_KEYS  = ["Original","Gaussian Blur\n(manual kernel)","Sobel X\n(manual)",
                "Sobel Y\n(manual)","Sobel Magnitude","Laplacian of Gaussian\n(LoG)","Canny Edges"]
FILTER_CMAPS = [None,"gray","RdBu","RdBu","hot","gray","gray"]

def run_prediction(mdl, img_bgr):
    img   = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img   = tf.keras.applications.mobilenet_v2.preprocess_input(
              cv2.resize(img, IMG_SIZE).astype(np.float32))
    probs = mdl.predict(np.expand_dims(img,0), verbose=0)[0]
    idx   = int(np.argmax(probs))
    return CLASS_LABELS[idx], float(probs[idx]), probs

def _draw_filter_row(fig, gs, flt_results):
    for col, (key, cmap) in enumerate(zip(FILTER_KEYS, FILTER_CMAPS)):
        ax = fig.add_subplot(gs[0, col]); img = flt_results.get(key)
        if img is not None:
            ax.imshow(img) if img.ndim==3 else ax.imshow(img, cmap=cmap or "gray")
        ax.set_title(key, color="white", fontsize=8, pad=4); ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_edgecolor("#444")

def _draw_prediction_panel(fig, gs, label, confidence, probs):
    lc = CLASS_COLORS[CLASS_LABELS.index(label)]
    ax = fig.add_subplot(gs[1,0:3]); ax.set_facecolor("#0f0f1a"); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
    ax.text(0.5,0.80,"Prediction",ha="center",color="#888",fontsize=11,transform=ax.transAxes)
    ax.text(0.5,0.58,label,ha="center",color=lc,fontsize=24,fontweight="bold",transform=ax.transAxes)
    ax.text(0.5,0.38,f"Confidence: {confidence*100:.1f}%",ha="center",color="white",fontsize=14,transform=ax.transAxes)
    ring=ax.inset_axes([0.72,0.05,0.25,0.55]); theta=np.linspace(0,2*np.pi*confidence,100)
    ring.plot(np.cos(theta),np.sin(theta),color=lc,linewidth=6); ring.plot(1,0,"o",color=lc,markersize=8)
    ring.add_patch(plt.Circle((0,0),1,color="#222",zorder=0)); ring.set_xlim(-1.3,1.3); ring.set_ylim(-1.3,1.3); ring.axis("off")
    ring.text(0,0,f"{confidence*100:.0f}%",ha="center",va="center",color="white",fontsize=11,fontweight="bold")

def _draw_probability_bars(fig, gs, probs):
    ax=fig.add_subplot(gs[1,3:7]); ax.set_facecolor("#0f0f1a")
    # Show only top 10 classes for readability
    top_idx  = np.argsort(probs)[::-1][:10]
    top_prob = probs[top_idx]
    top_lbl  = [CLASS_LABELS[j] for j in top_idx]
    y_pos    = np.arange(len(top_idx))
    bars=ax.barh(y_pos,top_prob,color=[CLASS_COLORS[j] for j in top_idx],alpha=0.85,height=0.6,edgecolor="none")
    for bar,prob in zip(bars,top_prob):
        ax.text(bar.get_width()+0.01,bar.get_y()+bar.get_height()/2,f"{prob*100:.1f}%",va="center",color="white",fontsize=10)
    ax.set_yticks(y_pos); ax.set_yticklabels(top_lbl,color="white",fontsize=10)
    ax.set_xlim(0,1.15); ax.set_xlabel("Probability",color="#888",fontsize=10)
    ax.set_title("Class Probabilities",color="white",fontsize=11); ax.tick_params(colors="white"); ax.tick_params(axis="x",labelsize=9)
    for s in ax.spines.values(): s.set_visible(False)

def visualize_prediction(img_bgr, mdl, save=False):
    img_bgr = cv2.resize(img_bgr, IMG_SIZE)
    flt_res = apply_all_filters(img_bgr)
    label, confidence, probs = run_prediction(mdl, img_bgr)
    fig = plt.figure(figsize=(20,11)); fig.patch.set_facecolor("#1a1a2e")
    gs  = gridspec.GridSpec(2,7,figure=fig,hspace=0.35,wspace=0.18,top=0.92,bottom=0.08,left=0.03,right=0.97)
    _draw_filter_row(fig, gs, flt_res)
    _draw_prediction_panel(fig, gs, label, confidence, probs)
    _draw_probability_bars(fig, gs, probs)
    fig.suptitle("Automated Plant Disease Classifier — Filter Visualization",color="white",fontsize=14,fontweight="bold")
    if save:
        out="outputs/full_prediction_visualization.png"
        plt.savefig(out,dpi=130,bbox_inches="tight",facecolor=fig.get_facecolor()); print(f"[✓] Saved {out}")
    plt.show(); print(f"\n[✓] Prediction: {label}  ({confidence*100:.1f}% confidence)")
    return label, confidence, probs

print("[✓] visualize_prediction ready")

In [ ]:
# ── Choose your image ────────────────────────────────────────────────────────
# Option A: Upload from your machine
# from google.colab import files
# uploaded = files.upload()
# img_bgr  = cv2.imread(list(uploaded.keys())[0])

# Option B: Use a dataset image
# img_bgr = cv2.imread(str(DATA_DIR / "Tomato_healthy" / "xxx.jpg"))

# Option C: Synthetic demo (default)
canvas = np.zeros((224,224,3), dtype=np.uint8)
canvas[:,:,1]=100; canvas[40:80,40:80]=[60,40,20]; canvas[130:180,100:160]=[50,30,10]
canvas  = cv2.add(canvas, np.random.randint(0,25,canvas.shape,dtype=np.uint8))
img_bgr = cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR)

print("[i] Using synthetic demo image — swap in a real leaf image for meaningful results.")
visualize_prediction(img_bgr, model, save=True)